# Day 4: ETL - Load & The Full Process

This is the final day of our SQL/ETL sprint! We'll complete the ETL process by focusing on the 'L' (Load). We'll also see how `pandas` and `SQL` can be used together seamlessly.

### 1. Setup and Imports

In [1]:
import pandas as pd
import sqlite3

print("Ready to Load!")

Ready to Load!


### 2. Core Concept: The Pandas <> SQL Bridge

You now know the two most important functions for this bridge:

1.  **SQL to Pandas (Extract):** 
    `df = pd.read_sql_query("SELECT * FROM my_table", conn)`

2.  **Pandas to SQL (Load):** 
    `df.to_sql("new_table_name", conn)`

Today, we will master the second one.

### 3. (T) Task 1: Re-create our Transformed DataFrame

Let's quickly run the extraction and transformation steps from yesterday to get our `df_transformed` ready.

In [2]:
# Step 1: Extract
df = pd.read_csv('raw_hr_data.csv')

# Step 2: Transform
df_transformed = df.copy()
df_transformed['salary'] = df_transformed['salary_str'].replace(r'[$,]', '', regex=True).astype(float)
df_transformed['job_title'] = df_transformed['job_title'].str.title()
df_transformed['hire_date'] = pd.to_datetime(df_transformed['hire_date'])
df_transformed[['first_name', 'last_name']] = df_transformed['full_name'].str.split(' ', n=1, expand=True)
df_transformed = df_transformed.drop(['full_name', 'salary_str'], axis=1)

print("Clean DataFrame is ready:")
df_transformed.head()

Clean DataFrame is ready:


,employee_id,job_title,hire_date,salary,first_name,last_name
0,101,Data Scientist,2021-05-10,120000.0,Sarah,Jenkins
1,102,Sr. Analyst,2020-03-15,85000.0,Mike,T.
2,103,Data Engineer,2022-01-20,130000.0,Leo,Kim
3,104,Data Scientist,2021-05-10,125000.0,Anjali,Gupta
4,105,Product Manager,2019-11-30,110000.0,Mark,O'Brien


### 4. (L) Task 2: Load DataFrame to SQL

This is the 'Load' step. We'll create a *new* database, `etl_warehouse.db`, to be our clean data warehouse. Then, we use `df.to_sql()` to save our DataFrame as a new SQL table.

Key parameters for `to_sql`:
- `name`: The name of the SQL table you want to create.
- `con`: The `sqlite3` connection object.
- `if_exists`: What to do if the table already exists?
  - `'fail'` (default): Raise an error.
  - `'replace'`: Drop the old table and create a new one.
  - `'append'`: Add the DataFrame rows to the existing table.
- `index=False`: We don't want to save the pandas index (0, 1, 2...) as a column in our SQL table.

In [3]:
# Create connection to our new 'Data Warehouse'
conn_warehouse = sqlite3.connect('db/etl_warehouse.db')

# LOAD the data!
df_transformed.to_sql(
    name='clean_employees', 
    con=conn_warehouse, 
    if_exists='replace', 
    index=False
)

print("Successfully loaded transformed data into 'etl_warehouse.db' in table 'clean_employees'")

Successfully loaded transformed data into 'etl_warehouse.db' in table 'clean_employees'


### 5. Task 3: Verify the Load

How do we know it worked? We do the *reverse*! We use `pd.read_sql_query` to read *from* our new warehouse database to check the data.

In [4]:
print("--- Verifying data in etl_warehouse.db ---")

df_from_warehouse = pd.read_sql_query("SELECT * FROM clean_employees", conn_warehouse)

conn_warehouse.close() # We can close the connection now

df_from_warehouse.head()

--- Verifying data in etl_warehouse.db ---


,employee_id,job_title,hire_date,salary,first_name,last_name
0,101,Data Scientist,2021-05-10 00:00:00,120000.0,Sarah,Jenkins
1,102,Sr. Analyst,2020-03-15 00:00:00,85000.0,Mike,T.
2,103,Data Engineer,2022-01-20 00:00:00,130000.0,Leo,Kim
3,104,Data Scientist,2021-05-10 00:00:00,125000.0,Anjali,Gupta
4,105,Product Manager,2019-11-30 00:00:00,110000.0,Mark,O'Brien


In [5]:
# Check the data types. 
# Note: SQLite doesn't have a dedicated 'datetime' type, so pandas reads it back as an object (string). 
# This is normal. When you read it back into pandas, you just convert it again if needed.
df_from_warehouse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   employee_id  5 non-null      int64  
 1   job_title    5 non-null      object 
 2   hire_date    5 non-null      object 
 3   salary       5 non-null      float64
 4   first_name   5 non-null      object 
 5   last_name    5 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 372.0+ bytes


### 6. Challenge: Full ETL Process Review

Let's put it all together in one function.

**Your Task:** Create a function `run_etl(csv_file, db_file)` that:
1.  **Extracts** data from `csv_file`.
2.  **Transforms** the data (clean salary and title).
3.  **Loads** the clean data into a table called `final_report` in the `db_file`.

In [6]:
def run_etl(csv_file, db_file, table_name):
    print(f"Starting ETL process for {csv_file}...")
    
    # 1. EXTRACT
    df = pd.read_csv(csv_file)
    print(f"Extracted {len(df)} rows.")
    
    # 2. TRANSFORM
    df['salary'] = df['salary_str'].replace(r'[$,]', '', regex=True).astype(float)
    df['job_title'] = df['job_title'].str.title()
    # Add a new transformation: Get bonus (10% of salary)
    df['bonus'] = df['salary'] * 0.10
    df = df.drop(['salary_str'], axis=1) # Drop old column
    print("Data transformed (bonus column added).")
    
    # 3. LOAD
    conn = sqlite3.connect(db_file)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    print(f"Successfully loaded data into {db_file}, table={table_name}.")
    
    return df

# Let's run our new function!
final_df = run_etl('raw_hr_data.csv', 'db/challenge_warehouse.db', 'final_report')

final_df.head()

Starting ETL process for raw_hr_data.csv...
Extracted 5 rows.
Data transformed (bonus column added).
Successfully loaded data into db/challenge_warehouse.db, table=final_report.


,employee_id,full_name,job_title,hire_date,salary,bonus
0,101,Sarah Jenkins,Data Scientist,2021-05-10,120000.0,12000.0
1,102,Mike T.,Sr. Analyst,2020-03-15,85000.0,8500.0
2,103,Leo Kim,Data Engineer,2022-01-20,130000.0,13000.0
3,104,Anjali Gupta,Data Scientist,2021-05-10,125000.0,12500.0
4,105,Mark O'Brien,Product Manager,2019-11-30,110000.0,11000.0


### AI Reset - SQL Sprint Complete!

Congratulations! You have successfully completed the 4-day SQL and ETL sprint.

You have learned the full lifecycle:
1.  **Day 1:** Basic SQL (`CREATE`, `INSERT`, `SELECT`, `WHERE`).
2.  **Day 2:** Advanced SQL (`UPDATE`, `DELETE`, `JOIN`, `GROUP BY`).
3.  **Day 3:** **(E)xtract** from CSVs/SQL and **(T)ransform** with `pandas`.
4.  **Day 4:** **(L)oad** data into a new DB with `to_sql` and reviewed the full process.

This is an incredibly valuable skill set, and you're now well-equipped to handle data both in memory (Pandas) and in a persistent database (SQL).